In [ ]:

import os
import json
import random
import time
import gc
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    get_linear_schedule_with_warmup,
    BitsAndBytesConfig
)
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_recall_fscore_support,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ============================================================================
# CONFIGURATION
# ============================================================================

# Model Selection - Choose your Qwen model
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Options:
# "Qwen/Qwen2.5-0.5B-Instruct"  # Smallest, fastest
# "Qwen/Qwen2.5-1.5B-Instruct"  # Balanced
# "Qwen/Qwen2.5-3B-Instruct"    # Better performance
# "Qwen/Qwen2.5-7B-Instruct"    # Best (requires more GPU)

# Data paths
TRAIN_PATH = "build_jsonl/build_train.jsonl"
DEV_PATH = "build_jsonl/build_dev.jsonl"
TEST_PATH = "build_jsonl/build_test.jsonl"

OUT_DIR = "qwen_legal_classification"
os.makedirs(OUT_DIR, exist_ok=True)

# Basic settings
SEED = 42
MAX_SEQ_LENGTH = 512  # Qwen can handle longer sequences
BATCH_SIZE = 2  # Adjust based on GPU memory
GRADIENT_ACCUMULATION_STEPS = 4

# Quantization (QLoRA for memory efficiency)
USE_QLORA = True
QLORA_BITS = 4
QLORA_COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# LoRA configuration
LORA_R = 64  # LoRA rank
LORA_ALPHA = 128
LORA_DROPOUT = 0.1

# Training hyperparameters
NUM_EPOCHS = 20
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.05

# Loss weights
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
LABEL_SMOOTHING = 0.02
PROTO_WEIGHT = 0.1

# Class imbalance handling
USE_WEIGHTED_SAMPLER = True
MINORITY_BOOST = 3.0

# Label definitions
LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER",
    "ARG_RESPONDENT", "ANALYSIS", "STA", "PRE_RELIED",
    "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(LABELS)

MINORITY_CLASSES = ["RLC", "ISSUE", "STA", "RATIO", "PRE_RELIED", "PRE_NOT_RELIED", "RPC"]
minority_ids = [label2id[label] for label in MINORITY_CLASSES if label in label2id]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================================
# UTILITIES
# ============================================================================

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_jsonl(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                data.append(json.loads(s))
    return data

def extract_data(docs, max_sents=64):
    all_sents, all_labels, doc_ids = [], [], []
    for doc in docs:
        doc_id = doc.get("id", "")
        sents, labs = [], []
        
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs = [label2id.get(l, label2id["NONE"]) for l in doc["labels"]]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs = [label2id.get(l, label2id["NONE"]) for l in doc["annotation"]]
        elif "annotations" in doc:
            for a in doc.get("annotations", []):
                for item in a.get("result", []):
                    val = item.get("value", {})
                    text = val.get("text", "").strip()
                    labs_list = val.get("labels", ["NONE"])
                    if text:
                        sents.append(text)
                        labs.append(label2id.get(labs_list[0], label2id["NONE"]))
        
        if len(sents) > max_sents:
            sents = sents[:max_sents]
            labs = labs[:max_sents]
        
        if sents and labs and len(sents) == len(labs):
            all_sents.append(sents)
            all_labels.append(labs)
            doc_ids.append(doc_id)
    
    return all_sents, all_labels, doc_ids

def compute_detailed_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0, labels=range(NUM_LABELS)
    )
    
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    minority_mask = np.isin(y_true, minority_ids)
    if minority_mask.sum() > 0:
        minority_true = y_true[minority_mask]
        minority_pred = y_pred[minority_mask]
        minority_f1 = f1_score(minority_true, minority_pred, average='macro', zero_division=0)
    else:
        minority_f1 = 0.0
    
    return {
        'macro_f1': float(macro_f1),
        'weighted_f1': float(weighted_f1),
        'minority_macro_f1': float(minority_f1),
        'per_class_f1': {id2label[i]: float(f1[i]) for i in range(NUM_LABELS)},
        'per_class_support': {id2label[i]: int(support[i]) for i in range(NUM_LABELS)},
    }

# ============================================================================
# PROTOTYPE MANAGER
# ============================================================================

class ClassPrototypeManager:
    def __init__(self):
        self.prototypes = None
        self.fitted = False

    def fit(self, embeddings, labels):
        embeddings = np.asarray(embeddings)
        labels = np.asarray(labels)
        D = embeddings.shape[1]
        protos = np.zeros((NUM_LABELS, D), dtype=np.float32)
        
        for k in range(NUM_LABELS):
            mask = labels == k
            if mask.sum() > 0:
                protos[k] = embeddings[mask].mean(axis=0)
            else:
                protos[k] = np.random.randn(D).astype(np.float32) * 1e-3
        
        self.prototypes = protos
        self.fitted = True
        print(f"[Prototypes] Fitted with shape {protos.shape}")

    def get_all_tensor(self, device=None):
        if not self.fitted:
            raise RuntimeError("Prototypes not fitted")
        t = torch.tensor(self.prototypes, dtype=torch.float32)
        if device is not None:
            t = t.to(device)
        return t

# ============================================================================
# DATASET
# ============================================================================

class QwenLegalDataset(Dataset):
    def __init__(self, docs_sents, docs_labels, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.docs_sents = docs_sents
        self.docs_labels = docs_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.docs_sents)

    def __getitem__(self, idx):
        sentences = self.docs_sents[idx]
        labels = self.docs_labels[idx]
        
        # Create a document context with sentence markers
        doc_text = ""
        for i, sent in enumerate(sentences):
            doc_text += f"[S{i}] {sent} "
        
        return {
            "text": doc_text.strip(),
            "sentences": sentences,
            "labels": labels,
            "num_sents": len(sentences)
        }

def collate_qwen_batch(batch, tokenizer, max_length=MAX_SEQ_LENGTH):
    texts = [b["text"] for b in batch]
    all_sentences = [b["sentences"] for b in batch]
    all_labels = [b["labels"] for b in batch]
    num_sents_list = [b["num_sents"] for b in batch]
    
    # Tokenize documents
    encoding = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    
    # Tokenize individual sentences for classification
    max_sents = max(num_sents_list)
    all_sent_encodings = []
    
    for sents in all_sentences:
        sent_enc = tokenizer(
            sents,
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        
        # Pad to max_sents
        if len(sents) < max_sents:
            pad_size = max_sents - len(sents)
            sent_enc['input_ids'] = torch.cat([
                sent_enc['input_ids'],
                torch.zeros(pad_size, sent_enc['input_ids'].size(1), dtype=torch.long)
            ])
            sent_enc['attention_mask'] = torch.cat([
                sent_enc['attention_mask'],
                torch.zeros(pad_size, sent_enc['attention_mask'].size(1), dtype=torch.long)
            ])
        
        all_sent_encodings.append(sent_enc)
    
    # Stack sentence encodings
    sent_input_ids = torch.stack([e['input_ids'] for e in all_sent_encodings])
    sent_attention_mask = torch.stack([e['attention_mask'] for e in all_sent_encodings])
    
    # Prepare labels
    labels_padded = torch.full((len(batch), max_sents), -100, dtype=torch.long)
    for i, labs in enumerate(all_labels):
        labels_padded[i, :len(labs)] = torch.tensor(labs, dtype=torch.long)
    
    return {
        'doc_input_ids': encoding['input_ids'],
        'doc_attention_mask': encoding['attention_mask'],
        'sent_input_ids': sent_input_ids,
        'sent_attention_mask': sent_attention_mask,
        'labels': labels_padded,
        'num_sents': torch.tensor(num_sents_list, dtype=torch.long)
    }

# ============================================================================
# QWEN MODEL FOR CLASSIFICATION
# ============================================================================

class QwenLegalClassifier(nn.Module):
    def __init__(self, model_name=QWEN_MODEL_NAME, num_labels=NUM_LABELS, 
                 use_qlora=USE_QLORA, dropout=0.3):
        super().__init__()
        
        # Load Qwen model
        if use_qlora:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=QLORA_COMPUTE_DTYPE,
            )
            self.qwen = AutoModelForCausalLM.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
            self.qwen = prepare_model_for_kbit_training(self.qwen)
        else:
            self.qwen = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto",
                trust_remote_code=True
            )
        
        # Apply LoRA
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            task_type=TaskType.CAUSAL_LM,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        )
        self.qwen = get_peft_model(self.qwen, lora_config)
        self.qwen.print_trainable_parameters()
        
        self.hidden_size = self.qwen.config.hidden_size
        
        # Classification head with context aggregation
        self.context_aggregator = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=self.hidden_size,
                nhead=8,
                dim_feedforward=self.hidden_size * 2,
                dropout=dropout,
                batch_first=True
            ),
            num_layers=2
        )
        
        # Prototype layer
        self.proto_proj = nn.Linear(self.hidden_size, self.hidden_size)
        
        # Final classifier
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.hidden_size, self.hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.hidden_size // 2, num_labels)
        )
    
    def get_sentence_embeddings(self, input_ids, attention_mask):
        """Extract embeddings for each sentence"""
        B, S, L = input_ids.shape
        
        # Flatten for processing
        input_ids_flat = input_ids.view(B * S, L)
        attention_mask_flat = attention_mask.view(B * S, L)
        
        # Get embeddings from Qwen
        with torch.no_grad():
            outputs = self.qwen.model(
                input_ids=input_ids_flat,
                attention_mask=attention_mask_flat,
                output_hidden_states=True
            )
        
        # Use last hidden state, average over sequence
        last_hidden = outputs.hidden_states[-1]  # [B*S, L, H]
        
        # Mean pooling
        mask_expanded = attention_mask_flat.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_embeddings = torch.sum(last_hidden * mask_expanded, 1)
        sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
        sent_embeddings = sum_embeddings / sum_mask
        
        # Reshape back
        sent_embeddings = sent_embeddings.view(B, S, -1)
        
        return sent_embeddings
    
    def forward(self, doc_input_ids, doc_attention_mask, sent_input_ids, 
                sent_attention_mask, num_sents, prototypes=None):
        B, S, L = sent_input_ids.shape
        
        # Get sentence-level embeddings
        sent_emb = self.get_sentence_embeddings(sent_input_ids, sent_attention_mask)
        
        # Add prototype information if available
        if prototypes is not None:
            proto_proj = self.proto_proj(sent_emb)
            proto_proj_norm = F.normalize(proto_proj, dim=-1)
            proto_norm = F.normalize(prototypes, dim=-1)
            proto_sim = torch.matmul(proto_proj_norm, proto_norm.t())
            proto_context = torch.matmul(proto_sim, prototypes)
            sent_emb = sent_emb + 0.1 * proto_context
        
        # Create attention mask for transformer
        sent_mask = torch.arange(S, device=sent_emb.device).unsqueeze(0) < num_sents.unsqueeze(1)
        
        # Apply context aggregation
        contextualized = self.context_aggregator(
            sent_emb,
            src_key_padding_mask=~sent_mask
        )
        
        # Classify each sentence
        logits = self.classifier(contextualized)
        
        return logits, sent_emb

# ============================================================================
# LOSS FUNCTION
# ============================================================================

def focal_loss(logits, labels, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA, label_smoothing=LABEL_SMOOTHING):
    """Focal loss for handling class imbalance"""
    ce = F.cross_entropy(logits, labels, reduction='none', label_smoothing=label_smoothing)
    pt = torch.exp(-ce)
    focal = alpha * (1 - pt) ** gamma * ce
    return focal.mean()

def compute_loss(logits, labels, sent_emb, prototypes, proto_weight=PROTO_WEIGHT):
    """Combined loss with prototype alignment"""
    # Flatten
    logits_flat = logits.view(-1, NUM_LABELS)
    labels_flat = labels.view(-1)
    
    # Mask padding
    mask = labels_flat != -100
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device)
    
    logits_masked = logits_flat[mask]
    labels_masked = labels_flat[mask]
    
    # Main classification loss
    cls_loss = focal_loss(logits_masked, labels_masked)
    
    # Prototype alignment loss
    if prototypes is not None and proto_weight > 0:
        sent_emb_flat = sent_emb.view(-1, sent_emb.size(-1))[mask]
        sent_norm = F.normalize(sent_emb_flat, dim=-1)
        proto_norm = F.normalize(prototypes, dim=-1)
        proto_sim = torch.matmul(sent_norm, proto_norm.t())
        proto_loss = F.cross_entropy(proto_sim * 5.0, labels_masked)
        
        total_loss = cls_loss + proto_weight * proto_loss
    else:
        total_loss = cls_loss
    
    return total_loss

# ============================================================================
# TRAINER
# ============================================================================

class QwenTrainer:
    def __init__(self, model, tokenizer, prototype_manager, device=DEVICE):
        self.model = model.to(device)
        self.tokenizer = tokenizer
        self.prototype_manager = prototype_manager
        self.device = device
    
    def _build_train_loader(self, dataset):
        if not USE_WEIGHTED_SAMPLER:
            return DataLoader(
                dataset, 
                batch_size=BATCH_SIZE, 
                shuffle=True,
                collate_fn=lambda b: collate_qwen_batch(b, self.tokenizer)
            )
        
        # Compute sample weights
        major_labels = []
        for doc_labels in dataset.docs_labels:
            if doc_labels:
                doc_counter = Counter(doc_labels)
                major_label = doc_counter.most_common(1)[0][0]
                major_labels.append(major_label)
            else:
                major_labels.append(0)
        
        counts = np.bincount(major_labels, minlength=NUM_LABELS)
        inv_freq = 1.0 / (counts + 1e-6)
        weights = inv_freq[np.array(major_labels)].copy()
        
        # Boost minority classes
        for doc_idx, major_label in enumerate(major_labels):
            if major_label in minority_ids:
                weights[doc_idx] *= MINORITY_BOOST
        
        sampler = WeightedRandomSampler(
            torch.tensor(weights, dtype=torch.double),
            num_samples=len(weights),
            replacement=True
        )
        
        return DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            collate_fn=lambda b: collate_qwen_batch(b, self.tokenizer)
        )
    
    def train(self, train_dataset, dev_dataset, num_epochs=NUM_EPOCHS, lr=LR):
        print(" Starting Qwen training...")
        
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        train_loader = self._build_train_loader(train_dataset)
        
        total_steps = len(train_loader) * num_epochs
        warmup_steps = int(WARMUP_RATIO * total_steps)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        
        prototypes_tensor = self.prototype_manager.get_all_tensor(device=self.device)
        best_macro_f1 = -1.0
        best_ckpt = None
        history = []
        
        for epoch in range(1, num_epochs + 1):
            self.model.train()
            epoch_start = time.time()
            running_loss = 0.0
            n_samples = 0
            
            for step, batch in enumerate(train_loader):
                batch = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v 
                        for k, v in batch.items()}
                
                logits, sent_emb = self.model(
                    batch['doc_input_ids'],
                    batch['doc_attention_mask'],
                    batch['sent_input_ids'],
                    batch['sent_attention_mask'],
                    batch['num_sents'],
                    prototypes_tensor
                )
                
                loss = compute_loss(logits, batch['labels'], sent_emb, prototypes_tensor)
                loss = loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()
                
                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                
                mask = batch['labels'].view(-1) != -100
                n = mask.sum().item()
                running_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS * n
                n_samples += n
                
                if (step + 1) % 10 == 0:
                    print(f"  Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
            avg_loss = running_loss / max(1, n_samples)
            
            # Evaluate
            val_results = self.evaluate(dev_dataset)
            epoch_time = time.time() - epoch_start
            
            history.append({
                "epoch": epoch,
                "train_loss": avg_loss,
                "val_macro_f1": val_results["metrics"]["macro_f1"],
                "epoch_time_s": epoch_time
            })
            
            print(f"\nEpoch {epoch}/{num_epochs} | "
                  f"Train Loss: {avg_loss:.4f} | "
                  f"Val Macro-F1: {val_results['metrics']['macro_f1']:.4f} | "
                  f"Minority F1: {val_results['metrics']['minority_macro_f1']:.4f} | "
                  f"Time: {epoch_time:.1f}s\n")
            
            # Save best model
            if val_results["metrics"]["macro_f1"] > best_macro_f1:
                best_macro_f1 = val_results["metrics"]["macro_f1"]
                ckpt_path = os.path.join(OUT_DIR, f"best_epoch{epoch}_f1{best_macro_f1:.4f}.pt")
                torch.save({
                    "model_state_dict": self.model.state_dict(),
                    "epoch": epoch,
                    "macro_f1": best_macro_f1,
                }, ckpt_path)
                best_ckpt = ckpt_path
                print(f" Saved best model: {ckpt_path}\n")
        
        pd.DataFrame(history).to_csv(os.path.join(OUT_DIR, "training_history.csv"), index=False)
        return best_ckpt
    
    def evaluate(self, dataset):
        self.model.eval()
        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=lambda b: collate_qwen_batch(b, self.tokenizer)
        )
        
        prototypes_tensor = self.prototype_manager.get_all_tensor(device=self.device)
        all_preds, all_trues = [], []
        
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v 
                        for k, v in batch.items()}
                
                logits, _ = self.model(
                    batch['doc_input_ids'],
                    batch['doc_attention_mask'],
                    batch['sent_input_ids'],
                    batch['sent_attention_mask'],
                    batch['num_sents'],
                    prototypes_tensor
                )
                
                logits_flat = logits.view(-1, NUM_LABELS)
                labels_flat = batch['labels'].view(-1)
                mask = labels_flat != -100
                
                if mask.sum() == 0:
                    continue
                
                preds = torch.argmax(logits_flat[mask], dim=1).cpu().numpy()
                all_preds.extend(preds.tolist())
                all_trues.extend(labels_flat[mask].cpu().numpy().tolist())
        
        metrics = compute_detailed_metrics(all_trues, all_preds)
        
        cls_report = classification_report(
            [id2label[x] for x in all_trues],
            [id2label[x] for x in all_preds],
            digits=4,
            zero_division=0
        )
        
        return {
            "metrics": metrics,
            "classification_report": cls_report,
            "all_preds": all_preds,
            "all_trues": all_trues
        }

# ============================================================================
# MAIN
# ============================================================================

def main():
    set_seed()
    
    print(" Loading data...")
    train_docs = load_jsonl(TRAIN_PATH)
    dev_docs = load_jsonl(DEV_PATH)
    test_docs = load_jsonl(TEST_PATH)
    
    train_sents, train_labels, _ = extract_data(train_docs)
    dev_sents, dev_labels, _ = extract_data(dev_docs)
    test_sents, test_labels, _ = extract_data(test_docs)
    
    print(f"Dataset sizes - Train: {len(train_sents)}, Dev: {len(dev_sents)}, Test: {len(test_sents)}")
    
    # Load tokenizer
    print(f"\n Loading Qwen tokenizer from {QWEN_MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Compute prototypes
    print("\n Computing class prototypes...")
    print("Loading temporary Qwen model for prototype computation...")
    
    if USE_QLORA:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=QLORA_COMPUTE_DTYPE,
        )
        temp_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        temp_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            device_map="auto",
            trust_remote_code=True
        )
    
    temp_model.eval()
    
    # Extract embeddings for prototypes
    flat_train_sents = [s for doc in train_sents for s in doc]
    flat_train_labels = np.array([l for doc in train_labels for l in doc], dtype=np.int64)
    
    train_embs = []
    batch_size = 8
    
    with torch.no_grad():
        for i in range(0, len(flat_train_sents), batch_size):
            batch = flat_train_sents[i:i+batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            enc = {k: v.to(temp_model.device) for k, v in enc.items()}
            
            outputs = temp_model.model(**enc, output_hidden_states=True)
            last_hidden = outputs.hidden_states[-1]
            
            # Mean pooling
            mask_expanded = enc['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeddings = torch.sum(last_hidden * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            sent_emb = sum_embeddings / sum_mask
            
            train_embs.append(sent_emb.cpu().numpy())
            
            if (i // batch_size + 1) % 50 == 0:
                print(f"  Processed {i+len(batch)}/{len(flat_train_sents)} sentences")
    
    train_embs = np.vstack(train_embs)
    
    proto_mgr = ClassPrototypeManager()
    proto_mgr.fit(train_embs, flat_train_labels)
    
    del temp_model
    torch.cuda.empty_cache()
    gc.collect()
    
    # Create datasets
    train_dataset = QwenLegalDataset(train_sents, train_labels, tokenizer)
    dev_dataset = QwenLegalDataset(dev_sents, dev_labels, tokenizer)
    test_dataset = QwenLegalDataset(test_sents, test_labels, tokenizer)
    
    # Initialize model
    print(f"\n Initializing Qwen model: {QWEN_MODEL_NAME}")
    model = QwenLegalClassifier()
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n Total params: {total_params:,} | Trainable: {trainable_params:,} "
          f"({100.0 * trainable_params / total_params:.2f}%)")
    
    # Train
    trainer = QwenTrainer(model, tokenizer, proto_mgr)
    best_ckpt = trainer.train(train_dataset, dev_dataset)
    
    # Load best checkpoint
    if best_ckpt:
        print(f"\n Loading best checkpoint: {best_ckpt}")
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"], strict=False)
    
    # Final test evaluation
    print("\n FINAL TEST EVALUATION")
    test_results = trainer.evaluate(test_dataset)
    
    print(f"\nTest Macro-F1:     {test_results['metrics']['macro_f1']:.4f}")
    print(f" Minority Macro-F1: {test_results['metrics']['minority_macro_f1']:.4f}")
    print(f" Weighted F1:       {test_results['metrics']['weighted_f1']:.4f}")
    print("\n Test Classification Report:")
    print(test_results["classification_report"])
    
    print("\n Per-Class F1 Scores:")
    for cls, f1 in test_results['metrics']['per_class_f1'].items():
        support = test_results['metrics']['per_class_support'][cls]
        minority_marker = "🔴" if cls in MINORITY_CLASSES else "  "
        print(f"  {minority_marker} {cls:20s}: {f1:.4f} (n={support})")
    
    # Save results
    results_summary = {
        'model': QWEN_MODEL_NAME,
        'method': 'QLoRA + Prototype Learning',
        'lora_rank': LORA_R,
        **test_results['metrics']
    }
    pd.DataFrame([results_summary]).to_csv(
        os.path.join(OUT_DIR, "final_results.csv"), index=False
    )
    
    pred_df = pd.DataFrame({
        "true": [id2label[x] for x in test_results["all_trues"]],
        "pred": [id2label[x] for x in test_results["all_preds"]],
    })
    pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)
    
    print(f"\n Results saved to {OUT_DIR}/")
    print("🏆 Qwen-based Legal Classification Complete! ")

if __name__ == "__main__":
    main()